In [6]:
import requests
from bs4 import BeautifulSoup
import tqdm

In [7]:
url="http://forumias.com/blog/constitution-of-india"
html=requests.get(url).text
soup=BeautifulSoup(html,'html.parser')

In [8]:
all_a=soup.find_all('a')
all_a=[i for i in all_a if 'Article '  in i.text]

In [9]:
all_a[0].text

'Article 1 of Indian Constitution'

In [10]:
all_a[0].get_attribute_list('href')

['https://forumias.com/blog/article-1-of-indian-constitution-name-and-territory-of-the-union/']

In [11]:
all_a_urls={}
for i in all_a:
    all_a_urls[i.text]=i.get_attribute_list('href')[0]

In [12]:
all_a_urls

{'Article 1 of Indian Constitution': 'https://forumias.com/blog/article-1-of-indian-constitution-name-and-territory-of-the-union/',
 'Article 2 of Indian Constitution': 'https://forumias.com/blog/article-2-of-indian-constitution/',
 'Article 3 of Indian Constitution ': 'https://forumias.com/blog/article-3-of-indian-constitution/',
 'Article 4 of Indian Constitution ': 'https://forumias.com/blog/article-4-of-indian-constitution/',
 'Article 5 of Indian Constitution': 'https://forumias.com/blog/article-5-of-indian-constitution/',
 'Article 6 of Indian Constitution': 'https://forumias.com/blog/article-6-of-indian-constitution/',
 'Article 7 of Indian Constitution': 'https://forumias.com/blog/article-7-of-indian-constitution/',
 'Article 8 of Indian Constitution': 'https://forumias.com/blog/article-8-of-indian-constitution/',
 'Article 9 of Indian Constitution': 'https://forumias.com/blog/article-9-of-indian-constitution/',
 'Article 10 of Indian Constitution': 'https://forumias.com/blog/a

In [13]:
url=all_a_urls['Article 1 of Indian Constitution']
html=requests.get(url).text
soup=BeautifulSoup(html,'html.parser')

In [14]:
article=soup.find_all(class_='entry-content')
print("\n".join([x.text for x  in article[0].find_all('p')]))

1. Name and territory of the Union.—
(1) India, that is Bharat, shall be a Union of States.
(2) The States and the territories thereof shall be as specified in the First Schedule.
(3) The territory of India shall comprise—
(a) the territories of the States;
[(b) the Union territories specified in the First Schedule; and]
(c) such other territories as may be acquired


In [15]:
all_articles={}
failed_articles=[]
for article_name,url in tqdm.tqdm(zip(list(all_a_urls.keys()),list(all_a_urls.values())),total=len(all_a_urls)): 
    try:
        html=requests.get(url).text
        soup=BeautifulSoup(html,'html.parser')
        article=soup.find_all(class_='entry-content')
        all_articles[article_name]="\n".join([x.text for x  in article[0].find_all('p')])
    except Exception as E:
        print(f'Failed for article {article_name} with error {E}')
        failed_articles.append(article_name)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 468/468 [12:44<00:00,  1.63s/it]


In [16]:
import json

with open('../data/articles.json','w') as f:
    json.dump(all_articles,f)

In [20]:
import json
data={}
with open('../data/articles.json','r') as f:
    data=json.load(f)

In [21]:
len(all_articles)

468

In [22]:
from IPython.display import display,Markdown
display(Markdown("# Indian Penal Code"))

# Indian Penal Code

In [1]:
import requests
from bs4 import BeautifulSoup


res=requests.get("https://lawrato.com/indian-kanoon/ipc")
soup=BeautifulSoup(res.text,'html.parser')

In [2]:
sections=soup.find_all(class_='col-sm-12 arrow-list-fancy')

In [18]:
all_sections_urls={}
for i in sections:
    all_sections_urls[i.find('a').text]=i.find('a').get_attribute_list('href')[0]

In [39]:
import tqdm
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

all_sections = {}
failed_articles = []

def fetch_section(key, url):
    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')
    text = soup.find(class_='content-left').text.strip()
    return key, text

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {
        executor.submit(fetch_section, key, url): key
        for key, url in all_sections_urls.items()
    }

    for future in tqdm.tqdm(as_completed(futures), total=len(futures)):
        key = futures[future]
        try:
            key, result = future.result()
            if "Repealed" not in result:
                all_sections[key] = result
        except Exception as e:
            print(f'Failed for section {key} with error {e}')
            failed_articles.append(key)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 575/575 [02:37<00:00,  3.64it/s]


In [54]:
x={}
for i,j in all_sections.items():
    if "repealed" not in j.lower() and 'substituted' not in j.lower():
        x[i] = j

In [55]:
len(all_sections),len(x)

(575, 551)

In [56]:
import json

with open('../data/penal_code_sections.json','w') as f:
    json.dump(all_sections,f)